Step 4.1 — Load Data & Inspect Relevant Columns
Purpose:
Load the dataset and confirm which household/building variables are available and populated.

In [1]:
import pandas as pd
import numpy as np

# File path
DATA_PATH = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_step2-2_individual_financial_metrics.csv"

# Load dataset
df = pd.read_csv(DATA_PATH)

# Columns relevant to Step 4
step4_cols = [
    "NUMBER_HABITABLE_ROOMS",
    "NUMBER_HEATED_ROOMS",
    "GLAZED_AREA",
    "MULTI_GLAZE_PROPORTION",
    "TOTAL_FLOOR_AREA",
    "FLOOR_LEVEL",
    "FLAT_TOP_STOREY",
    "MAIN_HEATING_CATEGORY",
    "MAIN_FUEL",
    "MAINS_GAS_FLAG",
    "PROPERTY_TYPE",
    "BUILT_FORM"
]

# Quick completeness check
df[step4_cols].isna().mean().sort_values()

NUMBER_HABITABLE_ROOMS    0.000000
NUMBER_HEATED_ROOMS       0.000000
GLAZED_AREA               0.000000
MULTI_GLAZE_PROPORTION    0.000000
TOTAL_FLOOR_AREA          0.000000
PROPERTY_TYPE             0.000000
BUILT_FORM                0.000000
MAIN_HEATING_CATEGORY     0.042735
MAIN_FUEL                 0.042735
FLOOR_LEVEL               0.923077
FLAT_TOP_STOREY           0.923077
MAINS_GAS_FLAG            1.000000
dtype: float64

Step 4.2 — Rooms & Heating Intensity Metrics
Purpose:
Quantify how intensively each dwelling is heated relative to its size and layout.
Derived metrics:
Heated room ratio
Rooms per m² (proxy for density / compactness)


In [2]:
df["HEATED_ROOM_RATIO"] = (
    df["NUMBER_HEATED_ROOMS"] / df["NUMBER_HABITABLE_ROOMS"]
)

df["ROOMS_PER_100M2"] = (
    df["NUMBER_HABITABLE_ROOMS"] / df["TOTAL_FLOOR_AREA"]
) * 100

# Replace infinite values where division failed
df.replace([np.inf, -np.inf], np.nan, inplace=True)

df[[
    "NUMBER_HABITABLE_ROOMS",
    "NUMBER_HEATED_ROOMS",
    "HEATED_ROOM_RATIO",
    "ROOMS_PER_100M2"
]].describe()

,NUMBER_HABITABLE_ROOMS,NUMBER_HEATED_ROOMS,HEATED_ROOM_RATIO,ROOMS_PER_100M2
count,117.000000,117.000000,113.000000,117.000000
mean,4.914530,4.846154,0.985841,4.256223
std,2.090919,2.135962,0.105108,1.345545
min,0.000000,0.000000,0.000000,0.000000
25%,4.000000,3.000000,1.000000,3.529412
50%,5.000000,5.000000,1.000000,4.188482
75%,6.000000,6.000000,1.000000,5.084746
max,15.000000,15.000000,1.000000,8.823529


Step 4.3 — Glazing Metrics
Purpose:
Standardize glazing exposure so it can be compared across dwelling sizes.
You already have GLAZED_AREA — this step contextualizes it.


In [3]:
df["GLAZED_AREA_PER_M2"] = (
    df["GLAZED_AREA"] / df["TOTAL_FLOOR_AREA"]
)

df["PERCENT_GLAZED"] = df["GLAZED_AREA_PER_M2"] * 100

df[[
    "GLAZED_AREA",
    "TOTAL_FLOOR_AREA",
    "GLAZED_AREA_PER_M2",
    "PERCENT_GLAZED"
]].describe()

,GLAZED_AREA,TOTAL_FLOOR_AREA,GLAZED_AREA_PER_M2,PERCENT_GLAZED
count,117.000000,117.000000,117.000000,117.000000
mean,1.290598,129.239316,0.014270,1.426959
std,1.017602,66.060141,0.017244,1.724352
min,0.000000,42.000000,0.000000,0.000000
25%,1.000000,77.000000,0.005618,0.561798
50%,1.000000,122.000000,0.008333,0.833333
75%,1.000000,167.000000,0.014706,1.470588
max,4.000000,425.000000,0.078431,7.843137


Step 4.4 — Floor Level & Flat Indicators
Purpose:
Make vertical position explicit and usable in analysis.


In [5]:
# Step 4.4 — Floor Level & Flat Indicators (Fixed)
# Purpose: Make vertical position explicit and usable, robust to missing/mixed data

# Is this dwelling a flat?
df["IS_FLAT"] = df["PROPERTY_TYPE"].str.contains("Flat", case=False, na=False)

# Is top-storey flat? (most data missing; fill missing as False)
df["IS_TOP_STOREY_FLAT"] = df["FLAT_TOP_STOREY"].fillna(False).astype(bool)

# Convert FLOOR_LEVEL to numeric, coercing errors to NaN
df["FLOOR_LEVEL_NUM"] = pd.to_numeric(df["FLOOR_LEVEL"], errors="coerce")

# Simplify floor level categories, filling missing as "Unknown"
df["FLOOR_LEVEL_SIMPLE"] = pd.cut(
    df["FLOOR_LEVEL_NUM"],
    bins=[-1, 0, 1, 2, 10],
    labels=["Ground", "First", "Second", "Third+"]
)

# Fill missing numeric categories with "Unknown"
df["FLOOR_LEVEL_SIMPLE"] = df["FLOOR_LEVEL_SIMPLE"].cat.add_categories(["Unknown"])
df["FLOOR_LEVEL_SIMPLE"].fillna("Unknown", inplace=True)

# Quick check
df[["PROPERTY_TYPE", "FLOOR_LEVEL", "FLOOR_LEVEL_NUM", "FLOOR_LEVEL_SIMPLE", "IS_FLAT", "IS_TOP_STOREY_FLAT"]].head(10)

/var/folders/57/_mkdbdyx3v15yrmh09sz1jmr0000gn/T/ipykernel_94395/3647602934.py:22: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["FLOOR_LEVEL_SIMPLE"].fillna("Unknown", inplace=True)


,PROPERTY_TYPE,FLOOR_LEVEL,FLOOR_LEVEL_NUM,FLOOR_LEVEL_SIMPLE,IS_FLAT,IS_TOP_STOREY_FLAT
0,Bungalow,NaN,NaN,Unknown,False,False
1,Bungalow,NaN,NaN,Unknown,False,False
2,House,NaN,NaN,Unknown,False,False
3,House,NaN,NaN,Unknown,False,False
4,House,NaN,NaN,Unknown,False,False
5,House,NaN,NaN,Unknown,False,False
6,House,NaN,NaN,Unknown,False,False
7,House,NaN,NaN,Unknown,False,False
8,House,NaN,NaN,Unknown,False,False
9,House,NaN,NaN,Unknown,False,False


Step 4.5 — Heating & Fuel Combinations
Purpose:
Create a single, interpretable heating archetype label for prioritization and reporting.


In [6]:
# Normalize text
df["MAIN_HEATING_CATEGORY_CLEAN"] = df["MAIN_HEATING_CATEGORY"].str.lower().str.strip()
df["MAIN_FUEL_CLEAN"] = df["MAIN_FUEL"].str.lower().str.strip()

# Combined heating-fuel label
df["HEATING_FUEL_COMBO"] = (
    df["MAIN_HEATING_CATEGORY_CLEAN"]
    + " | "
    + df["MAIN_FUEL_CLEAN"]
)

# Gas vs non-gas indicator
df["IS_GAS_HEATED"] = df["MAINS_GAS_FLAG"].astype(bool)

df[[
    "MAIN_HEATING_CATEGORY",
    "MAIN_FUEL",
    "HEATING_FUEL_COMBO",
    "IS_GAS_HEATED"
]].value_counts().head(10)

MAIN_HEATING_CATEGORY                           MAIN_FUEL                                     HEATING_FUEL_COMBO                                                                             IS_GAS_HEATED
heat pump with radiators or underfloor heating  electricity                                   heat pump with radiators or underfloor heating | electricity                                   True             32
boiler with radiators or underfloor heating     oil                                           boiler with radiators or underfloor heating | oil                                              True             27
electric storage heaters                        electricity                                   electric storage heaters | electricity                                                         True             15
community heating system                        biomass                                       community heating system | biomass                                          

Step 4.6 — Aggregate Building Characteristics Snapshot
Purpose:
Produce a compact summary table suitable for:
Your website
FDT briefing
Methodology appendix


In [7]:
building_characteristics_summary = {
    "Avg habitable rooms": df["NUMBER_HABITABLE_ROOMS"].mean(),
    "Avg heated rooms": df["NUMBER_HEATED_ROOMS"].mean(),
    "Avg heated room ratio": df["HEATED_ROOM_RATIO"].mean(),
    "Avg % glazed": df["PERCENT_GLAZED"].mean(),
    "% flats": df["IS_FLAT"].mean() * 100,
    "% top-storey flats": df["IS_TOP_STOREY_FLAT"].mean() * 100,
    "% gas heated": df["IS_GAS_HEATED"].mean() * 100
}

pd.Series(building_characteristics_summary).round(2)

Avg habitable rooms        4.91
Avg heated rooms           4.85
Avg heated room ratio      0.99
Avg % glazed               1.43
% flats                    7.69
% top-storey flats         3.42
% gas heated             100.00
dtype: float64

Step 4.7 — Save Updated Dataset
Purpose:
Persist Step 4 outputs so all downstream steps (5–9) can reuse them.


In [8]:
OUTPUT_PATH = DATA_PATH.replace(
    "step2-2_individual_financial_metrics",
    "step4_building_characteristics"
)

df.to_csv(OUTPUT_PATH, index=False)

OUTPUT_PATH

'/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_step4_building_characteristics.csv'